In [ ]:
# ── Connection — Variable Library VL_SALES_ORD  ───────────────────────────────
# Switched from VL_SD to VL_SALES_ORD to match the refactored Bronze pipeline.
# BRONZE_SCHEMA = 'dbo' aligns with NB_Bronze_SALES_ORD_v2 which writes to dbo. 
_vl = notebookutils.variableLibrary.getLibrary('VL_SALES_ORD')

SOURCE_LH_ABFSS    = _vl['SOURCE_LH_ABFSS'].strip()
SOURCE_SCHEMA      = 'dbo'

BRONZE_LH_ABFSS    = _vl['BRONZE_LH_ABFSS'].strip()
BRONZE_SCHEMA      = 'dbo'     # Bronze writes to dbo — must match here

SILVER_LH_ABFSS    = _vl['SILVER_LH_ABFSS'].strip()
MD_SILVER_LH_ABFSS = _vl['MD_SILVER_LH_ABFSS'].strip()
SILVER_SCHEMA      = 'dbo'

OPS_LH_ABFSS       = _vl['OPS_LH_ABFSS'].strip()
OPS_SCHEMA         = 'dbo'

PIPELINE_NAME    = 'SD_Silver'
PIPELINE_RUN_ID  = ''

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 3, Finished, Available, Finished, False)

In [2]:
%run ./NB_Config_SALES_ORD_v2

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 5, Finished, Available, Finished, True)

SO config loaded: ['VBAK', 'VBAP', 'VBEP', 'LIKP', 'LIPS']
Overwrite tables: TR_Backlog_2WEEKS, MAP_MPG_Sales_Org
SD desc config loaded: 25 tables


In [3]:
%run ./NB_Utils_Silver_v2

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 12, Finished, Available, Finished, True)

NB_Utils_Silver imports loaded
_ops_append + log_metrics ready
reorder_columns() ready
process_table() ready
apply_xrate() ready
apply_unit_conv() ready

── NB_Utils_Silver fully loaded ────────────────────────────────────────────
_apply_enrichments() ready

── NB_Utils_Silver_v2 fully loaded ─────────────────────────────────────────


In [4]:
# ── Pipeline run setup ───────────────────────────────────────────────────────
import uuid
from datetime import datetime

if not PIPELINE_RUN_ID:
    PIPELINE_RUN_ID = str(uuid.uuid4())

spark.conf.set('spark.sql.parquet.datetimeRebaseModeInWrite', 'CORRECTED')
spark.conf.set('spark.sql.parquet.datetimeRebaseModeInRead',  'CORRECTED')
spark.conf.set('spark.sql.adaptive.enabled', 'true')

start_time = datetime.utcnow()
print(f'Pipeline : {PIPELINE_NAME}')
print(f'Run ID   : {PIPELINE_RUN_ID}')
print(f'Started  : {start_time}')

# Desc tables owned by MD pipeline — desc_joins route reads to MD_SILVER_LH_ABFSS.
# SD-specific desc tables (TVAKT, TVAUT, etc.) are promoted from Bronze to Silver
# in the next cell, then found via SILVER_LH_ABFSS routing automatically.
MD_SILVER_TABLES = {
    # Original MD tables
    'T001W', 'T024E', 'T024', 'T001L', 'T001T', 'T161T', 'T156HT',
    # Customer group texts (TVV series — confirmed in MD Silver)
    'TVV1T', 'TVV2T', 'TVV3T', 'TVV4T', 'TVV5T',
    # Material group texts (TVM series — confirmed in MD Silver)
    'TVM1T', 'TVM2T', 'TVM3T', 'TVM4T', 'TVM5T',
    # Customer / vendor / material master
    'KNA1', 'MAKT', 'LFA1',
    # SD org / product hierarchy desc tables (loaded via MD pipeline)
    'TVKOT', 'TSPAT', 'TVKBT', 'T171T', 'T151T', 'T179T', 'T134T', 'T001',
}

# VBEP (schedule lines) and LIKP (delivery header) have no Material column —
# unit_conv reads MARM and joins on Material, so it would always fail for these tables.
SD_TABLE_CONFIGS['VBEP'].pop('enrich', None)
SD_TABLE_CONFIGS['LIKP'].pop('enrich', None)

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 13, Finished, Available, Finished, False)

Pipeline : SD_Silver
Run ID   : aad3abb4-96d1-4144-b113-e1581e304f75
Started  : 2026-09-21 22:36:23.575253


[{'type': 'xrate',
  'currency_col': 'Document_Currency',
  'date_col': 'Created_On',
  'amount_cols': ['Net_Value']},
 {'type': 'unit_conv',
  'qty_col': 'Net_Weight',
  'uom_col': 'Weight_Unit',
  'primary_unit': 'KG',
  'conv_unit': 'LB'},
 {'type': 'unit_conv',
  'qty_col': 'Net_Weight',
  'uom_col': 'Weight_Unit',
  'primary_unit': 'LB',
  'conv_unit': 'KG'}]

In [5]:
# ── SD Desc table Bronze→Silver promoter ─────────────────────────────────────
# SD-specific desc tables (TVAKT, TVAUT, etc.) are ingested into Bronze SD by
# NB_Bronze_SALES_ORD_v2. Silver's process_table only routes desc reads to
# SILVER_LH_ABFSS or MD_SILVER_LH_ABFSS — it has no Bronze routing.
# This cell promotes SD desc tables from Bronze into Silver (overwrite each run)
# so the main executor can find them via the normal SILVER_LH_ABFSS path.
#
# SAP stores language keys as single characters: 'E' = English (not 'EN').
# Filter applied here so Silver desc tables contain only English descriptions.

print('Promoting SD desc tables Bronze → Silver...')
_promoted, _skipped = [], []

for _tbl, _dcfg in SD_DESC_CONFIGS.items():
    _bronze_path = f'{BRONZE_LH_ABFSS}/{BRONZE_SCHEMA}/{_tbl}'
    _silver_path = f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{_tbl}'
    try:
        _df = (spark.read.format('delta').load(_bronze_path)
                    .filter(F.col('_IS_DELETED') == False))

        # Select only the columns declared in config (skip missing ones gracefully)
        _cols = [c for c in _dcfg['silver_cols'] if c in _df.columns]
        _df = _df.select(_cols)

        # SAP language code: 'E' = English (single char, not 'EN')
        if 'SPRAS' in _df.columns:
            _df = _df.filter(F.col('SPRAS') == 'E')

        (_df.write.format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .save(_silver_path))
        spark.sql(
            f"CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.{_tbl} "
            f"USING DELTA LOCATION '{_silver_path}'"
        )
        _n = _df.count()
        _promoted.append(_tbl)
        print(f'  {_tbl}: {_n:,} rows → Silver')
    except Exception as _e:
        _skipped.append(_tbl)
        print(f'  {_tbl}: skipped ({_e})')

print(f'\nPromoted : {len(_promoted)} tables')
print(f'Skipped  : {len(_skipped)} tables')

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 14, Finished, Available, Finished, False)

Promoting SD desc tables Bronze → Silver...
  TVAKT: 209 rows → Silver
  TVAUT: 940 rows → Silver
  TVFKT: 129 rows → Silver
  T176T: 50 rows → Silver
  T014T: 29 rows → Silver
  T024B: 296 rows → Silver
  T691T: 276 rows → Silver
  TVLVT: 54 rows → Silver
  TVFST: 20 rows → Silver
  TVST: 335 rows → Silver
  TVROT: 292 rows → Silver
  TMVFT: 39 rows → Silver
  T178T: 49 rows → Silver
  CEPCT: 317 rows → Silver
  T006A: 354 rows → Silver
  TVEPT: 112 rows → Silver
  TVLST: 42 rows → Silver
  TVSTT: 394 rows → Silver
  TINCT: 33 rows → Silver
  TPRIT: 4 rows → Silver
  T173T: 25 rows → Silver
  T148T: 13 rows → Silver
  TVAPT: 493 rows → Silver
  TLGRT: 35 rows → Silver
  T023T: 5,028 rows → Silver

Promoted : 25 tables
Skipped  : 0 tables


In [6]:
# ── Single-pass concurrent executor ─────────────────────────────────────────
# All SD tables are independent — single concurrent pass.
# Desc reads for MD_SILVER_TABLES route to MD_SILVER_LH_ABFSS.
# SK joins always route to MD_SILVER_LH_ABFSS (Material_SK, Plant_SK, Customer_SK).
results = []

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {
        pool.submit(
            process_table, name, SD_TABLE_CONFIGS[name],
            md_silver_lh_abfss=MD_SILVER_LH_ABFSS,
            md_silver_tables=MD_SILVER_TABLES
        ): name
        for name in SD_TABLE_CONFIGS
    }
    for f in as_completed(futures):
        try:
            results.append(f.result())
        except Exception as e:
            name = futures[f]
            print(f'  FAILED {name}: {e}')
            log_metrics(name, 'SILVER', {'rows_in': 0, 'rows_active': 0,
                        'null_pk_dropped': 0, 'dupes_removed': 0, 'rows_out': 0},
                        warnings=[{'metric': 'error', 'value': 0, 'message': str(e)}])

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 15, Finished, Available, Finished, False)


── VBAK ──

── VBAP ──

── VBEP ──

── LIKP ──

── LIPS ──
  Bronze rows    : 50,584
  Bronze rows    : 70,894
  Bronze rows    : 229,457
  Bronze rows    : 358,003
  Bronze rows    : 30,662
  Active rows    : 358,003
  Active rows    : 50,584
  Active rows    : 30,662
  Active rows    : 229,457
  Active rows    : 70,894
  Dupes removed  : 2 (quarantined to OPS)
  Dupes removed  : 8 (quarantined to OPS)
  Dupes removed  : 10 (quarantined to OPS)
  Skipped desc join TVLKT ([PATH_NOT_FOUND] Path does not exist: abfss://c1b617bd-849f-472c-97c3-ddfdbbb58da7@onelake.dfs.fabric.microsoft.com/c2dee42e-0de4-4af3-98ba-f3de0de1536c/Tables/dbo/TVLKT.)
  Skipped desc join TVSBT ([PATH_NOT_FOUND] Path does not exist: abfss://c1b617bd-849f-472c-97c3-ddfdbbb58da7@onelake.dfs.fabric.microsoft.com/c2dee42e-0de4-4af3-98ba-f3de0de1536c/Tables/dbo/TVSBT.)
  Skipped desc join TVSBT ([PATH_NOT_FOUND] Path does not exist: abfss://c1b617bd-849f-472c-97c3-ddfdbbb58da7@onelake.dfs.fabric.microsoft.com/c2dee42e

In [7]:
# ── Config-driven post-write enrichments ─────────────────────────────────────
# Runs sequentially so all Silver tables are written before enrichment reads them.
print('\nPost-write enrichments...')
for _tbl, _cfg in SD_TABLE_CONFIGS.items():
    if _cfg.get('enrich'):
        _apply_enrichments(
            _tbl, _cfg,
            silver_path=f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{_tbl}',
            sk_col=_cfg['sk_col'],
        )
print('Enrichments complete.')

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 16, Finished, Available, Finished, False)


Post-write enrichments...

  Enriching VBAK (1 steps)...
    4 cols enriched — 50,582 rows

  Enriching VBAP (3 steps)...
    6 cols enriched — 229,449 rows

  Enriching LIPS (2 steps)...
    2 cols enriched — 70,894 rows
Enrichments complete.


In [8]:
# ── BW Backlog overwrite load ────────────────────────────────────────────────
# SAP_BW_TR_Backlog_2WEEKS is planned for SD but not yet available in the source.
# Uncomment when the extract is confirmed in SOURCE_LH_ABFSS.
#
# print(f'\nLoading {BW_BACKLOG_TABLE} (overwrite)...')
# bw_source_path = f'{SOURCE_LH_ABFSS}/{SOURCE_SCHEMA}/{BW_BACKLOG_TABLE}'
# bw_silver_path = f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{BW_BACKLOG_TABLE}'
# bw_df = (spark.read.format('delta').load(bw_source_path)
#               .withColumn('_PIPELINE_NAME',   F.lit(PIPELINE_NAME))
#               .withColumn('_PIPELINE_RUN_ID', F.lit(PIPELINE_RUN_ID))
#               .withColumn('_UPDATED_AT',      F.current_timestamp()))
# bw_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(bw_silver_path)
# spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.{BW_BACKLOG_TABLE} USING DELTA LOCATION '{bw_silver_path}'")
# n = spark.read.format('delta').load(bw_silver_path).count()
# print(f'  {BW_BACKLOG_TABLE} loaded — {n:,} rows')
print('BW Backlog load skipped — table not yet available in source.')

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 17, Finished, Available, Finished, False)

BW Backlog load skipped — table not yet available in source.


In [9]:
# ── MAP_MPG_Sales_Org overwrite load ─────────────────────────────────────────
# MAP_MPG_Sales_Org is a custom MPG→Sales_Organization mapping used in the OTIF
# Gold notebook. It lives in MD Silver (Map schema), not in the SD source.
# Commented out — uncomment and update source path when this table is confirmed
# in SOURCE_LH_ABFSS.
#
# print(f'\nLoading {MPG_MAPPING_TABLE} (overwrite)...')
# mpg_source_path = f'{SOURCE_LH_ABFSS}/{SOURCE_SCHEMA}/{MPG_MAPPING_TABLE}'
# mpg_silver_path = f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{MPG_MAPPING_TABLE}'
# mpg_df = (spark.read.format('delta').load(mpg_source_path)
#                .withColumn('_PIPELINE_NAME',   F.lit(PIPELINE_NAME))
#                .withColumn('_PIPELINE_RUN_ID', F.lit(PIPELINE_RUN_ID))
#                .withColumn('_UPDATED_AT',      F.current_timestamp()))
# mpg_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(mpg_silver_path)
# spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.{MPG_MAPPING_TABLE} USING DELTA LOCATION '{mpg_silver_path}'")
# n = spark.read.format('delta').load(mpg_silver_path).count()
# print(f'  {MPG_MAPPING_TABLE} loaded — {n:,} rows')
print('MAP_MPG_Sales_Org load skipped — table lives in MD Silver, not SD source.')

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 18, Finished, Available, Finished, False)

MAP_MPG_Sales_Org load skipped — table lives in MD Silver, not SD source.


In [10]:
# ── Summary ──────────────────────────────────────────────────────────────────
duration = (datetime.utcnow() - start_time).total_seconds()

print('\n' + '=' * 80)
print('SD SILVER — SUMMARY')
print('=' * 80)
print(f'{"Table":<22} {"In":>8} {"Active":>8} {"NullPK":>8} {"Dupes":>8} {"Out":>8}  Gold')
print('-' * 80)
for r in results:
    print(f'{r["table"]:<22} {r["rows_in"]:>8,} {r["rows_active"]:>8,} '
          f'{r["null_pk_dropped"]:>8,} {r["dupes_removed"]:>8,} {r["rows_out"]:>8,}  → {r["gold"]}')
print('-' * 80)
for tbl in (BW_BACKLOG_TABLE, MPG_MAPPING_TABLE):
    path = f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{tbl}'
    try:
        n = spark.read.format('delta').load(path).count()
        print(f'{tbl:<22} {"(overwrite)":>43}  {n:>8,} rows')
    except Exception:
        print(f'{tbl:<22}  NOT FOUND')
print('-' * 80)
print(f'Run ID   : {PIPELINE_RUN_ID}')
print(f'Duration : {duration:.1f}s')

StatementMeta(, ccab8ad2-d7fd-4f02-a096-dc86d7865058, 19, Finished, Available, Finished, False)


SD SILVER — SUMMARY
Table                        In   Active   NullPK    Dupes      Out  Gold
--------------------------------------------------------------------------------
VBEP                    358,003  358,003        0       10  357,993  → Fact_Schedule_Line
LIKP                     30,662   30,662        0        0   30,662  → Fact_Delivery_Item
VBAK                     50,584   50,584        0        2   50,582  → Fact_Sales_Doc_Item
VBAP                    229,457  229,457        0        8  229,449  → Fact_Sales_Doc_Item
LIPS                     70,894   70,894        0        0   70,894  → Fact_Delivery_Item
--------------------------------------------------------------------------------
TR_Backlog_2WEEKS       NOT FOUND
MAP_MPG_Sales_Org       NOT FOUND
--------------------------------------------------------------------------------
Run ID   : aad3abb4-96d1-4144-b113-e1581e304f75
Duration : 418.9s
